**Changes from previous experiments**
- Removed `trans_day`
- Removed `trans_weekday`
- Removed velocity features
- Removed UID2 (`card1 + addr1 + D1n`)
- Uses LightGBM native categorical support
- Uses early stopping
- Performs threshold tuning
- Reports engineered UID feature importance

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score

In [2]:
train = pd.read_pickle("../data/processed/train_optimized.pkl")
train = train.copy()
print(train.shape)
train.head()

(590540, 434)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [3]:
print(train.shape)

(590540, 434)


# Feature Engineering
### Core UID Features
Removed trans_day, trans_weekday
These were proven useless during ablation testing.
Kept trans_hour
Only used temporarily to compute `uid_mean_hour`. It is dropped before training.
UID is built from
```
card1 + addr1
```
Aggregations are calculated using an expanding window with `.shift(1)` to avoid target leakage.

In [4]:
train = train.sort_values("TransactionDT").reset_index(drop=True)
train["trans_hour"] = ((train["TransactionDT"] % 86400) // 3600).astype(np.int8)
train["uid"] = (train["card1"].astype(str) + "_" + train["addr1"].astype(str))
train["uid"] = train["uid"].fillna("unknown_addr")
uid_grp = train.groupby("uid")["TransactionAmt"]
train["uid_transaction_count"] = uid_grp.cumcount()
train["uid_mean_amount"] = uid_grp.transform(lambda x: x.shift(1).expanding().mean())
train["uid_std_amount"] = uid_grp.transform(lambda x: x.shift(1).expanding().std())
train["uid_amount_ratio"] = (   train["TransactionAmt"] / train["uid_mean_amount"])
train["uid_amount_zscore"] = ((train["TransactionAmt"] - train["uid_mean_amount"]) / train["uid_std_amount"])
train["uid_mean_hour"] = (train.groupby("uid")["trans_hour"].transform(lambda x: x.shift(1).expanding().mean()))
for col in ["uid_mean_amount", "uid_std_amount", "uid_amount_ratio", "uid_amount_zscore", "uid_mean_hour",]:
    train[col] = train[col].fillna(-1)

# Prepare Training Data
Removed feature families: Velocity features, UID2 (card1, addr1, D1n)
These reduced validation performance.
Dropped before modeling: isFraud, TransactionID, uid, trans_hour

In [5]:
drop_cols = ["isFraud", "TransactionID", "uid", "trans_hour",]
X = train.drop(columns=drop_cols)
y = train["isFraud"]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,stratify=y,random_state=42,)

In [6]:
cat_cols = X_train.select_dtypes(include=["category", "object"]).columns.tolist()
for col in cat_cols:
    X_train[col] = X_train[col].astype("category")
    X_test[col] = X_test[col].astype("category")

`Train LightGBM`\
Uses
- Early stopping
- Native categorical handling
- Validation monitoring

In [7]:
model = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, num_leaves=31, random_state=42,n_jobs=-1,)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], categorical_feature=cat_cols, callbacks=[ lgb.early_stopping(50), lgb.log_evaluation(0),],)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.632695 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 41007
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 437
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034989 -> initscore=-3.317101
[LightGBM] [Info] Start training from score -3.317101
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's binary_logloss: 0.0553052


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,1000
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


`Threshold Tuning`\
Evaluate several probability thresholds and compare
- Recall
- Precision
- F1 Score

In [12]:
def threshold_tuning(model):
    y_proba = model.predict_proba(X_test)[:, 1]
    thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50,]
    print(f"{'Thresh':>6} | {'Recall':>6} | {'Precision':>9} | {'F1':>6}")
    print("-" * 45)
    for thresh in thresholds:
        preds = (y_proba >= thresh).astype(int)
        recall = recall_score(y_test, preds)
        precision = precision_score(y_test, preds)
        f1 = f1_score(y_test, preds)
        print(f"{thresh:6.2f} | "f"{recall:6.3f} | "f"{precision:9.3f} | "f"{f1:6.3f}")

In [13]:
threshold_tuning(model)

Thresh | Recall | Precision |     F1
---------------------------------------------
  0.10 |  0.796 |     0.599 |  0.683
  0.15 |  0.758 |     0.733 |  0.745
  0.20 |  0.726 |     0.809 |  0.765
  0.25 |  0.693 |     0.859 |  0.767
  0.30 |  0.669 |     0.895 |  0.766
  0.40 |  0.615 |     0.932 |  0.741
  0.50 |  0.569 |     0.954 |  0.712


`UID Feature Importance`
Check whether the engineered UID features are actually being used by LightGBM.

In [9]:
feat_imp = pd.Series(model.feature_importances_,index=X_train.columns,).sort_values(ascending=False)
uid_features = ["uid_transaction_count", "uid_mean_amount","uid_std_amount","uid_amount_ratio","uid_amount_zscore","uid_mean_hour",]
print("\nUID Feature Rankings:\n")
for feature in uid_features:
    if feature in feat_imp.index:
        rank = feat_imp.rank(ascending=False)[feature]
        print(f"{feature}: "f"rank {int(rank)} "f"(importance {feat_imp[feature]:.0f})")


UID Feature Rankings:

uid_transaction_count: rank 8 (importance 697)
uid_mean_amount: rank 9 (importance 641)
uid_std_amount: rank 14 (importance 506)
uid_amount_ratio: rank 19 (importance 377)
uid_amount_zscore: rank 22 (importance 361)
uid_mean_hour: rank 10 (importance 637)


In [10]:
original_cols = set(pd.read_pickle('../data/processed/train_optimized.pkl').columns)
print(type(original_cols))
current_cols = set(train.columns)
new_cols = current_cols - original_cols
print(f'{len(original_cols)}\n{len(current_cols)}\nnew columns added: {len(new_cols)}')
for col in sorted(new_cols):
    print(f' {col}')

<class 'set'>
434
442
new columns added: 8
 trans_hour
 uid
 uid_amount_ratio
 uid_amount_zscore
 uid_mean_amount
 uid_mean_hour
 uid_std_amount
 uid_transaction_count


| Version | Stage     | Model         | Changes                                                                            |     Best F1 |        Δ F1 |        AUC | Decision                      |
| ------- | --------- | ------------- | ---------------------------------------------------------------------------------- | ----------: | ----------: | ---------: | ----------------------------- |
| V1      | Baseline  | Random Forest | Raw processed dataset                                                              |   **0.602** |           — |          — | ✅ Baseline                    |
| V2      | Baseline  | Random Forest | Balanced class weights                                                             |   **0.570** |      -0.032 |          — | ❌ Reject                      |
| V3      | FE        | LightGBM      | Switched to LightGBM                                                               |       ~0.74 |       +0.14 |          — | ✅ Keep                        |
| V4      | FE        | LightGBM      | Threshold tuning                                                                   |      ~0.748 |       +0.02 |          — | ✅ Keep                        |
| V5      | FE        | LightGBM      | Added `trans_day`, `trans_hour`, `trans_weekday`                                   |     No gain |       0.000 |          — | ❌ Remove                      |
| V6      | FE        | LightGBM      | UID aggregation features                                                           |      ~0.748 |    Positive |          — | ✅ Keep                        |
| V7      | FE        | LightGBM      | Velocity features                                                                  |       0.743 |      -0.005 |          — | ❌ Remove                      |
| V8      | FE        | LightGBM      | UID2 (`card1+addr1+D1n`)                                                           |       0.696 |      -0.052 |          — | ❌ Reject                      |
| V9      | FE        | LightGBM      | Cleanup + native categoricals + early stopping                                     |  **0.7669** |      +0.019 | **0.9611** | ✅ Current LGBM Baseline       |
| V10     | Benchmark | XGBoost       | Tuned boosting parameters                                                          |  **0.7454** |     -0.0215 | **0.9578** | ⚠️ Competitive but below LGBM |
| V11     | Benchmark | CatBoost      | GPU + Lossguide + Depth=8 + L2 + Early Stopping                                    |  **0.7881** | **+0.0212** | **0.9672** | 🏆 Best Model                 |
| V12     | Kaggle FE | All Models    | Implemented Kaggle winner engineered features and saved `train_v10_winning_fe.pkl` | In Progress |           — |          — | 🚧 Active Research            |


In [ ]:
train.to_pickle('../data/processed/train_v9_engineered.pkl', protocol=4)
print(f"Saved: {train.shape[0]:,} rows × {train.shape[1]} columns")
print(f"File size: ~{train.memory_usage(deep=True).sum() / 1024**3:.2f} GB")
print("\nColumns in saved file:")
print(train.columns.tolist())

Saved: 590,540 rows × 442 columns
File size: ~0.95 GB

Columns in saved file:
['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'V61', 'V62', 'V63', 'V64', 'V65', 'V66', 'V67', 'V68', 'V69', 'V70', '